# Machine-learning trade model

This optional notebook is separate from the core rules-based workflow. It trains a chronological Ridge model and evaluates its predictions with the shared backtester.

In [ ]:
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next((path for path in candidates if (path / 'src' / 'trading').is_dir()), None)
if project_root is None:
    raise RuntimeError('Could not locate the parent project.')
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(project_root / 'Machine Learning Trade Model' / 'src'))
from ml_trade_model.modeling import walk_forward_predictions
from trading.config import load_config
from trading.data import load_prices
from trading.risk import portfolio_risk_diagnostics
from trading.strategy import weekly_momentum_weights
from trading.backtest import format_performance_report, run_backtest
from trading.visualization import plot_diagnostics, plot_periodic_returns
config = load_config(project_root / 'config.yaml')
prices = load_prices(project_root / 'data' / 'processed' / 'prices.parquet')


In [ ]:
predictions, quality = walk_forward_predictions(prices, config['benchmark'])
print('ML MODEL QUALITY')
print(f"Out-of-sample observations: {quality['observations']:,.0f}")
print(f"Prediction/realized-return correlation: {quality['correlation']:.3f}")
weights = weekly_momentum_weights(
    prices, config['benchmark'], config['top_n'], config['lookback_days'], config['skip_days'],
    config['volatility_days'], config['max_position_weight'], config['max_equity_exposure'],
    config['defensive_exposure'], config['stop_loss_pct'], config['drawdown_start'],
    config['drawdown_full_defense'], expected_returns=predictions,
    volatility_target=config['volatility_target']
)
ledger, holdings, trades = run_backtest(
    prices, weights, config['initial_capital'], config['commission_bps'],
    config['half_spread_bps'], config['slippage_bps'], config['market_impact_bps']
)
risk = portfolio_risk_diagnostics(holdings, prices, config['benchmark'])
print('ML-BIASED RANKING MODEL')
print(format_performance_report(ledger, config['initial_capital']))
print(f'Trades recorded: {len(trades):,}')


In [ ]:
plot_diagnostics(ledger, trades, risk, prices[config['benchmark']])
plot_periodic_returns(ledger, period='Y')
plot_periodic_returns(ledger, period='M')
plot_periodic_returns(ledger, period='5Y')


In [ ]:
trades.style.format({
    'execution_price': '${:,.3f}', 'notional': '${:,.3f}', 'estimated_cost': '${:,.3f}',
    'realized_profit_loss': '${:,.3f}', 'net_profit_loss': '${:,.3f}',
    'profit_loss_pct': '{:.3%}', 'cumulative_net_profit_loss': '${:,.3f}',
})
